In [37]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [38]:

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    shear_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest"
)

val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

In [39]:
train_data = train_datagen.flow_from_directory(
    'Face Mask Dataset/Train',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

val_data = val_datagen.flow_from_directory(
    'Face Mask Dataset/Validation',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

test_data = test_datagen.flow_from_directory(
    'Face Mask Dataset/Test',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

Found 1406 images belonging to 2 classes.


Found 800 images belonging to 2 classes.
Found 992 images belonging to 2 classes.


In [40]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam

In [41]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

for layer in base_model.layers[:-20]:
    layer.trainable = False

In [42]:
from tensorflow.keras.layers import BatchNormalization

In [43]:
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    BatchNormalization(),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

In [44]:
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [45]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [46]:


callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ReduceLROnPlateau(factor=0.3, patience=2)
]

In [47]:

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,264,385 (8.64 MB)

 Trainable params: 1,209,921 (4.62 MB)

 Non-trainable params: 1,054,464 (4.02 MB)

In [49]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=1,
    callbacks=callbacks
)

44/44 ━━━━━━━━━━━━━━━━━━━━ 51s 1s/step - accuracy: 0.9872 - loss: 0.0430 - val_accuracy: 0.9025 - val_loss: 0.2282 - learning_rate: 1.0000e-04


In [50]:
batch = next(train_data)
x, y = batch

pred = model.predict(x)

print("Predictions:", pred[:5])
print("Labels:", y[:5])

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Predictions: [[0.1716505 ]
 [0.05203767]
 [0.19953409]
 [0.999969  ]
 [0.99910736]]
Labels: [0. 0. 0. 1. 1.]


In [ ]:
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image

# Load model
# model = load_model("mask_detector_model.h5")

# Class labels (IMPORTANT: match your training order)
class_labels = ['mask', 'no_mask']

def predict_image(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array)[0][0]

    label = class_labels[1] if prediction > 0.5 else class_labels[0]
    confidence = prediction if prediction > 0.5 else 1 - prediction

    print(f"Prediction: {label}")
    print(f"Confidence: {confidence:.2f}")

# Example
# predict_image("test.jpg")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
Prediction: mask
Confidence: 0.82


In [55]:
predict_image("C:/Users/Jash/Downloads/Austin_Butler_at_the_2025_Cannes_Film_Festival_02.jpg")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Prediction: no_mask
Confidence: 0.98


In [56]:
predict_image("C:/Users/Jash/Downloads/716MYI1-ppL._AC_UF1000,1000_QL80_.jpg")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
Prediction: mask
Confidence: 0.82
